This is the third part of the [User guide](../../usage.qmd). [Part 1](../../data.qmd) describes the source data and one-match preprocessing; [part 2](../setup/possession_and_checkpoint_setup.ipynb) defines the segmentation and context labels.

The complete example follows **Ivan Vujica’s possession `8_774` in Melbourne City FC–Macarthur FC, 7 March 2025**. Vujica receives in his own third, carries and completes a pass to Luke Brattan. It is the first case in the [case studies](../07_case_studies/07_case_studies.ipynb).

Here the models distributed with PRESS evaluate the observed frames. The case studies use held-out predictions, so their numbers need not equal this deployment example.

## 1. Installation {#installation}

From a local PRESS checkout, create a Python 3.11 environment and install the package:

```bash
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -e .
```

To run the downloadable notebook, also install its notebook dependencies:

```bash
python -m pip install -e '.[vignettes]'
```

The five fitted models are included; no training is required. Set the original data location:

```bash
export SKILLCORNER_OPENDATA="/path/to/opendata"
```

## 2. Original match inputs {#prepare-model-inputs}

The loader expects the original match JSON, tracking JSONL and dynamic-event CSV in the match directory. It prepares velocities, possessions, context and actions in memory. No separate preprocessing command or precomputed Parquet file is required.

## 3. Select the match directory {#load-a-match}

The following cell selects the original public match and the individual possession illustrated in the report.


In [3]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PRESS import PRESS
from PRESS.visualization import plot_frame, plot_risk

opendata = Path(os.environ.get("SKILLCORNER_OPENDATA", Path.cwd().parent / "opendata"))
output_dir = Path(os.environ.get("PRESS_EXAMPLE_OUTPUT", "outputs/user_guide"))
output_dir.mkdir(parents=True, exist_ok=True)
figure_dir = Path("figures")
figure_dir.mkdir(exist_ok=True)
match_id = "2006229"
individual_id = "8_774"
match_dir = opendata / "data/matches" / match_id
print("Original SkillCorner match:", match_id)
print("Required files: match JSON, tracking JSONL and dynamic-event CSV")

Original SkillCorner match: 2006229
Required files: match JSON, tracking JSONL and dynamic-event CSV


In [4]:
opendata

## 4. Initialize the object {#initialize}

One `PRESS` object holds one match. It validates and copies these inputs, links team possessions from verified individual intervals, and creates empty result dictionaries. Model predictions are calculated only when requested.


In [6]:
press = PRESS.from_skillcorner(match_dir)
tracking = press.tracking_data
individuals = press.get_player_possessions()
actions = press.actions_data
print("Match metadata:", press.get_match_info())
print("Preparation:", press.get_preparation_report())
print("Stored predictions after initialization:")
print({component: len(frames) for component, frames in press.results.items()})

Match metadata: {'fps': 10.0, 'pitch_length': 105.0, 'pitch_width': 68.0, 'match_id': '2006229'}
Preparation: {'provider': 'SkillCorner', 'match_id': '2006229', 'fps': 10.0, 'n_tracking_frames': 59270, 'n_source_events': 4991, 'n_individual_possessions': 959, 'n_actions': 2627, 'player_velocity_window_frames': 7, 'ball_velocity_window_frames': 5, 'boundary_source': 'provider matched individual possession annotations'}
Stored predictions after initialization:
{'PPCF': 0, 'BALL_DRIVE_RISK': 0, 'ACTION_SELECTION': 0, 'HAZARD': 0, 'BCOS': 0, 'RESIDUAL_RISK': 0, 'XLOSS': 0}


The loader reads the original SkillCorner files and prepares velocities, possessions, action checkpoints and match context in memory. It requires no precomputed Parquet tables. For data already prepared by the caller, initialize `PRESS(tracking_data, ...)` directly; see the [initialization reference](../../api.qmd).

## 5. Select a possession and recover its checkpoints {#select-possession}

Select the individual possession by its ID and retrieve the linked team interval. `get_player_frames` selects the individual’s observed frames. `get_actions(possession_id=individual_id)` selects the individual’s canonical actions. Their `frame_start` values are the checkpoints; `valid_only=True` retains usable start states.


In [8]:
individual = press.get_player_possessions(individual_id)
team_id = individual.team_possession_id.item()
team = press.get_team_possessions(team_id)
frames = press.get_player_frames(individual_id, valid_only=True)
actions = press.get_actions(possession_id=individual_id, valid_only=True)
actions["elapsed_seconds"] = (
    actions.frame_start - individual.frame_start.item()
) / press.fps

print("Individual:", individual_id, "carrier:", individual.player_id.item())
print("Linked team possession:", team_id)
print("Observed individual frames:", int(frames.frame.min()), "to", int(frames.frame.max()))
print(actions[
    ["frame_start", "elapsed_seconds", "action", "previous_action"]
].to_string(index=False))


Individual: 8_774 carrier: away_13
Linked team possession: 190
Observed individual frames: 41434 to 41463
 frame_start  elapsed_seconds     action previous_action
       41434              0.0 ball_drive  pass_reception
       41444              1.0 ball_drive      ball_drive
       41463              2.9       pass      ball_drive


The three canonical checkpoints are the carry starts at 0 and 1 s and the pass release at 2.9 s. The short-tail start at 2 s is available only when requesting additional snapshots. Its definition, together with `previous_action`, is given in [part 2](../setup/possession_and_checkpoint_setup.ipynb#previous-action).

## 6. Plot any usable frame {#generic-frame}

Frame `41439` is 0.5 s after reception, inside the first carry. It is not a canonical checkpoint. The getter returns the observed tracking row; `get_state` extracts the numerical player/ball state and `get_frame_context` resolves the current event context.


In [10]:
frame_id = int(individual.frame_start.item() + round(0.5 * press.fps))
frame = press.get_frame(frame_id)
state = press.get_state(frame_id)
context = press.get_frame_context(frame_id)
print("Selected frame:", frame_id)
print("Carrier:", state["player"])
print("Context:", context)
print("Elapsed individual-control time:", press.get_elapsed_time(frame_id))
assert frame_id not in actions.frame_start.to_list()

fig, ax = plot_frame(
    press, frame_id, show=False,
    title="Ivan Vujica · observed frame 41439 · 0.5 s after reception",
)
fig.savefig(figure_dir / "user_guide_generic_frame.png", dpi=170)
fig.savefig(figure_dir / "user_guide_generic_frame.svg")
plt.close(fig)


Selected frame: 41439
Carrier: away_13
Context: {'start_type': 'pass_reception', 'previous_action': 'pass_reception', 'completed_canonical_drives': 0, 'time_since_start': 0.5, 'source': 'individual_possession'}
Elapsed individual-control time: 0.5


![A generic observed frame inside Vujica’s first carry, drawn by the package plot function.](../../figures/user_guide_generic_frame.png)

## 7. Inspect the canonical checkpoints {#plot-checkpoints}

Plot the same observed pitch at the three action starts. At this stage no predictions have been calculated. The last panel shows the pass-release state; the ball’s subsequent flight belongs to the execution of that pass.


In [12]:
fig, axes = plt.subplots(1, len(actions), figsize=(18, 5.4))
for index, (ax, row) in enumerate(zip(np.atleast_1d(axes), actions.itertuples())):
    plot_frame(
        press, int(row.frame_start), ax=ax, show=False,
        legend=index == len(actions) - 1,
        title=f"t={row.elapsed_seconds:.1f} s · {row.action} · frame {row.frame_start}",
        title_fontsize=10,
    )
fig.savefig(figure_dir / "user_guide_canonical_frames.png", dpi=170)
fig.savefig(figure_dir / "user_guide_canonical_frames.svg")
plt.close(fig)


![Observed states at the two canonical carry starts and the terminal pass release.](../../figures/user_guide_canonical_frames.png)

## 8. Calculate h and xLoss at the generic frame {#compute-frame}

`compute_risk_frame(..., include_residual=True)` runs all five fitted components: pass execution, carry execution, pass destination, pass–carry choice and conditional residual risk. It calculates only the preliminary effective-hazard BCOS required as a feature by those fitted models. The final transport descriptor is available separately through `compute_bcos()` and is not also evaluated for every risk checkpoint. The fitted xT surfaces and physical parameters are loaded from the model artifacts distributed with PRESS.

The current score is supplied explicitly and is relative to Vujica’s team. Elapsed time is recovered from the observed individual interval. h describes loss on the current action; xLoss adds continuation risk to predict the terminal individual-possession outcome.


In [14]:
outputs = press.compute_risk_frame(
    frame_id,
    include_residual=True,
)
h = outputs["HAZARD"]["Hazard"]
xloss = outputs["XLOSS"]["xLoss"]
band = outputs["HAZARD"]["band"]
print(f"Frame {frame_id}: h={h:.2%}, xLoss={xloss:.2%}, band={band}")
print("Elapsed time used:", press.get_elapsed_time(frame_id))

fig, ax = plot_risk(
    press, frame_id, field="hazard", show=False,
    title=f"Ivan Vujica · frame {frame_id} · h={h:.1%} · xLoss={xloss:.1%}",
)
fig.savefig(figure_dir / "user_guide_generic_risk.png", dpi=170)
fig.savefig(figure_dir / "user_guide_generic_risk.svg")
plt.close(fig)


Frame 41439: h=34.47%, xLoss=54.97%, band=High
Elapsed time used: 0.5


![h and xLoss from the distributed models, evaluated at the generic frame.](../../figures/user_guide_generic_risk.png)

## 9. See where the object stores results {#stored-results}

Results are indexed by **component, then frame ID**. Complete risk calculations retain compact outputs by default: destination-weighted pass success/failure, carry risk, action choice, BCOS area, h with its band, residual risk and xLoss. Each frame refers to shared settings through `configuration_id`. Physical/numerical settings and xT inputs are stored in `press.result_configurations`; fitted coefficients, spline bases and tree models remain in `press.result_models`.

Use `get_result_configuration(frame_id)` and `get_result_model(component, frame_id)` to inspect the settings and fitted model actually used, including custom parameters. To retain spatial fields, defender contributions and feature records for inspection or plotting, use `keep_details=True` in the complete risk call. Separately called component methods retain their detailed outputs.


| Level | Ball-drive example | BCOS example |
|---|---|---|
| **Model** | Estimated `lambda_0`, `s`, linear/spline coefficients, knots and basis transformations. | No separately fitted predictor; its default dispersion comes from the ball-drive model. |
| **Configuration** | Defender maximum speed, accelerations, integration step and forecast horizon. | Separate carrier/defender maximum speeds, dispersion input, accelerations, spatial radius, grid and numerical orders. |
| **Result** | Loss probability for this frame; optional defender contributions and survival curve. | Accessible area for this frame; optional accessibility surface and interception diagnostics. |

For example, a ball-drive `physics_kwargs={"v_max": 6.5}` changes defender motion without changing the estimated coefficients or spline bases. A `model={"s": 0.35}` changes the model parameter while retaining omitted coefficients and bases. Both produce a new result for the selected frame. Pitch control uses its own motion settings; BCOS distinguishes carrier `v_bc_max` from defender `v_def_max`. Per-call overrides do not automatically change other components. [Model, configuration and result: full contract and defaults](../../api.qmd#model-configuration-result).


The same model/configuration/result convention applies to individual component calls; retrieve their physical/numerical settings with `get_result_configuration(frame_id, component="BALL_DRIVE_RISK")`, for example. Fitted pass-control `lambda_control`, `s` and isotonic calibration remain in the pass-risk model.

For frame inspection, use `keep_details=True`: spatial maps, defender contributions and the action-selection feature record remain available. For many possessions, use the default `keep_details=False`: only compact predictions and context remain, together with references to shared settings/models. Risk and action-probability annotations can use compact results; spatial maps require retained details. Tree feature names/order/categories stay in the fitted model, while their values at a frame are calculation inputs, not fitted parameters.


In [16]:
print("Computed frames per component:")
print({component: sorted(frames) for component, frames in press.results.items()})
print("Stored h record:", press.results["HAZARD"][frame_id])
print("Stored xLoss record:", press.results["XLOSS"][frame_id])

# Getters recover copies of the existing calculations.
h_record = press.get_hazard_result(frame_id)
xloss_record = press.get_xloss_result(frame_id)
pass_field = press.get_ppcf_result(frame_id)
drive = press.get_ball_drive_risk_result(frame_id)
print("Expected pass success:", pass_field["Expected pass success"])
print("Spatial details retained:", pass_field["details_available"])
print("Physical settings:", press.get_result_configuration(frame_id)["BALL_DRIVE_RISK"])
print("Fitted drive parameters:", press.get_result_model("BALL_DRIVE_RISK", frame_id).parameters)
print("Stored carry-loss probability:", drive["Ball drive risk"])


Computed frames per component:
{'PPCF': [41439], 'BALL_DRIVE_RISK': [41439], 'ACTION_SELECTION': [41439], 'HAZARD': [41439], 'BCOS': [41439], 'RESIDUAL_RISK': [41439], 'XLOSS': [41439]}
Stored h record: {'Hazard': 0.3447480208919997, 'band': 'High', 'Drive risk': 0.1514981451397548, 'Pass failure expected': 0.6238434451412735, 'rppcf': 0.37615655485872646, 'configuration_id': '0542f4084782bfabd846aa8b1645bcb10cd6eea7ae121cef957d8ca45a40e549', 'details_available': False}
Stored xLoss record: {'xLoss': 0.5497234308134128, 'Hazard': 0.3447480208919997, 'Residual loss probability': 0.4088423252105713, 'time_since_start': 0.5, 'configuration_id': '0542f4084782bfabd846aa8b1645bcb10cd6eea7ae121cef957d8ca45a40e549', 'details_available': False}
Expected pass success: 0.37615655485872646
Spatial details retained: False
Physical settings: {'t_press': 1.0, 'physics_kwargs': {'v_max': 7.0, 'a_linear': 7.0, 'a_turn': 7.0, 'preserve_initial_overspeed': True, 'pitch_length': 105.0, 'pitch_width': 68.0

The getters do not run the models again. `get_results()` retrieves a copy of the full result cache; use component-specific getters when only one output is needed. [Getter reference](../../reference/getters.qmd).

## 10. Calculate the complete canonical profile {#compute-checkpoints}

Evaluate the same models at all three checkpoints. `include_snapshots=False` explicitly selects the canonical sequence. The score remains unchanged over this 2.9 s possession; match-clock minutes are read separately from each tracking frame.


In [18]:
profile = press.compute_risk_player_possession(
    individual_id,
    include_snapshots=False,
    include_residual=True,
)
profile = profile.merge(
    actions[["frame_start", "elapsed_seconds", "action"]].rename(
        columns={"frame_start": "frame_id"}),
    on="frame_id", validate="one_to_one",
)
profile = profile.sort_values("frame_id").reset_index(drop=True)
print(profile[
    ["frame_id", "elapsed_seconds", "action", "p_pass", "Hazard", "xLoss", "band"]
].to_string(index=False, float_format=lambda value: f"{value:.4f}"))
assert profile.frame_id.to_list() == actions.frame_start.to_list()


 frame_id  elapsed_seconds     action  p_pass  Hazard  xLoss   band
    41434           0.0000 ball_drive  0.4541  0.2911 0.4349 Medium
    41444           1.0000 ball_drive  0.3358  0.3385 0.5392   High
    41463           2.9000       pass  0.8182  0.0937 0.1275    Low


### Scores assigned to the checkpoints {#checkpoint-scores}

The table reports P(pass), h, xLoss and the h band at each action start. The figure below annotates those calculated scores on the observed pitch. These are state-level probabilities, not the player score E−O reported in the case studies.


In [20]:
fig, axes = plt.subplots(1, len(profile), figsize=(18, 5.4))
for index, (ax, row) in enumerate(zip(np.atleast_1d(axes), profile.itertuples())):
    plot_risk(
        press, int(row.frame_id), field="hazard", ax=ax, show=False,
        legend=index == len(profile) - 1,
        title=(f"t={row.elapsed_seconds:.1f} s · {row.action} · {row.band}\n"
               f"h={row.Hazard:.1%} · xLoss={row.xLoss:.1%}"),
        title_fontsize=10,
    )
fig.savefig(figure_dir / "user_guide_checkpoint_scores.png", dpi=170)
fig.savefig(figure_dir / "user_guide_checkpoint_scores.svg")
plt.close(fig)

fig, ax = plot_frame(
    press, int(individual.frame_receipt.item()), show=False,
    title="Luke Brattan · matched reception · frame 41476",
)
fig.savefig(figure_dir / "user_guide_completed_pass.png", dpi=170)
fig.savefig(figure_dir / "user_guide_completed_pass.svg")
plt.close(fig)


![Calculated h, xLoss and h bands at the canonical checkpoints.](../../figures/user_guide_checkpoint_scores.png)

![The matched reception by Luke Brattan confirms that Vujica completed the final pass. This is an outcome frame, not another checkpoint in Vujica’s possession.](../../figures/user_guide_completed_pass.png)

The generic-frame prediction at 0.5 s is stored too, but does not become a fourth canonical action. Canonical h profiles and their retrospective interpretation are described in [h and bands](../06_composed_hazard_and_bands/06_composed_risk_and_bands.ipynb). An xLoss prediction remains a terminal individual-outcome probability at each state.

## 11. Save and reload the results {#save-results}

Save a compact checkpoint table and the complete calculated arrays. `save_results` writes `metadata.json` and `arrays.npz`; it does not serialize the tracking inputs or an entire Python object. This tutorial explicitly replaces its own previous result export when rerun.


In [22]:
profile.to_csv(output_dir / "checkpoint_scores.csv", index=False)
export_dir = press.save_results(output_dir / "match_results", overwrite=True)
print("Saved files:", sorted(path.name for path in export_dir.iterdir()))

restored = PRESS.from_skillcorner(match_dir)
restored.load_results(export_dir)
restored_profile = pd.DataFrame([
    {
        "frame_id": int(frame),
        "h": restored.get_hazard_result(int(frame))["Hazard"],
        "xLoss": restored.get_xloss_result(int(frame))["xLoss"],
    }
    for frame in profile.frame_id
])
print(restored_profile.to_string(index=False))
np.testing.assert_allclose(restored_profile.h, profile.Hazard, rtol=0, atol=0)
np.testing.assert_allclose(restored_profile.xLoss, profile.xLoss, rtol=0, atol=0)
print("Reloaded scores agree exactly with the calculated checkpoint scores.")


Saved files: ['arrays.npz', 'metadata.json']
 frame_id        h    xLoss
    41434 0.291118 0.434927
    41444 0.338485 0.539169
    41463 0.093706 0.127518
Reloaded scores agree exactly with the calculated checkpoint scores.


Loading validates the input and model fingerprints before restoring the cached calculations. The restored object uses the same match tables, context columns and model settings. [Persistence reference](../../reference/getters.qmd).

For other tasks, continue with the [model explanations](../../index.qmd), [held-out benchmarks](../../benchmark.qmd) or [case studies](../07_case_studies/07_case_studies.ipynb).